## 0.Importación librerias

In [42]:
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
import os
import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import mlflow.sklearn

In [43]:
cwd = Path.cwd().resolve()
try:
    HERE = next(c for c in [cwd, *cwd.parents]
                if (c / ".git").exists() or (c / ".dvc").exists())
except StopIteration:
    raise FileNotFoundError(
        "No encuentro la raíz del repo (.git/.dvc). "
        "Abre el notebook dentro del proyecto agropredict-colombia."
    )

DATOS_PROCESADOS = HERE / "data" / "processed"
ARCHIVO_LIMPIO = DATOS_PROCESADOS / "maiz_preprocesado.parquet"

In [44]:
load_dotenv(HERE / "config" / ".env", override=True)
MLFLOW_URI = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5001")
MLFLOW_EXPERIMENT = os.getenv("MLFLOW_EXPERIMENT", "agropredict-maiz-rendimiento")

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print(f"MLflow apuntando a: {MLFLOW_URI}")
print(f"Experimento: {MLFLOW_EXPERIMENT}")

2026/11/07 11:15:08 INFO mlflow.tracking.fluent: Experiment with name 'agropredict-maiz-rendimiento' does not exist. Creating a new experiment.


MLflow apuntando a: http://3.80.56.180:5000
Experimento: agropredict-maiz-rendimiento


## 1.Configuración del experimento

In [45]:
# Tipo de información histórica: "estacional" (mismo semestre año anterior)
#                               o "secuencial" (semestre inmediatamente anterior)
TIPO_REZAGO = "estacional"

# Separación temporal de los datos
ANIOS_ENTRENAMIENTO = [2019, 2020, 2021, 2022, 2023]
ANIO_VALIDACION = 2024
ANIO_PRUEBA = 2025

# Semilla para reproducibilidad.
SEMILLA = 42

# Resumen de la configuración activa
print("Configuración del experimento:")
print(f"  Tipo de rezago : {TIPO_REZAGO}")
print(f"  Entrenamiento  : {ANIOS_ENTRENAMIENTO}")
print(f"  Validación     : {ANIO_VALIDACION}")
print(f"  Prueba         : {ANIO_PRUEBA}")
print(f"  Semilla        : {SEMILLA}")

Configuración del experimento:
  Tipo de rezago : estacional
  Entrenamiento  : [2019, 2020, 2021, 2022, 2023]
  Validación     : 2024
  Prueba         : 2025
  Semilla        : 42


In [46]:
# Cargamos la tabla de features generada en el notebook 03_featuring
ARCHIVO_FEATURES = DATOS_PROCESADOS / f"features_{TIPO_REZAGO}.parquet"
base = pd.read_parquet(ARCHIVO_FEATURES)

print(f"Features cargadas: {base.shape[0]:,} filas ({TIPO_REZAGO})")
base.head()

Features cargadas: 14,935 filas (estacional)


,cod_departamento,departamento,cod_municipio,municipio,sistema_productivo,anio,semestre_b,area_cosechada,produccion,rendimiento,t,rendimiento_rezago
0,05,Antioquia,05002,Abejorral,tradicional,2019,0,100.0,200.0,2.0,4038,NaN
1,05,Antioquia,05002,Abejorral,tradicional,2019,1,75.0,150.0,2.0,4039,NaN
2,05,Antioquia,05002,Abejorral,tradicional,2020,0,100.0,200.0,2.0,4040,2.0
3,05,Antioquia,05002,Abejorral,tradicional,2020,1,75.0,150.0,2.0,4041,2.0
4,05,Antioquia,05002,Abejorral,tradicional,2021,0,98.0,196.0,2.0,4042,2.0


## 2.Separación temporal de los datos

Dividimos los datos respetando la línea de tiempo, según la configuración definida
al inicio:

- **Entrenamiento:** 2019–2023
- **Validación:** 2024
- **Prueba:** 2025

Antes de separar, quitamos las filas sin rezago (las primeras apariciones de cada
municipio, que no tienen historia).

In [47]:
modelo = base.dropna(subset=["rendimiento_rezago"]).copy()
print(f"Filas utilizables (con rezago): {len(modelo):,} de {len(base):,}")

# Separación temporal según la configuración del experimento
train = modelo[modelo["anio"].isin(ANIOS_ENTRENAMIENTO)]
val   = modelo[modelo["anio"] == ANIO_VALIDACION]
test  = modelo[modelo["anio"] == ANIO_PRUEBA]

print(f"\nEntrenamiento ({ANIOS_ENTRENAMIENTO[0]}-{ANIOS_ENTRENAMIENTO[-1]}): {len(train):,} filas")
print(f"Validación ({ANIO_VALIDACION}): {len(val):,} filas")
print(f"Prueba ({ANIO_PRUEBA}): {len(test):,} filas")

Filas utilizables (con rezago): 12,158 de 14,935

Entrenamiento (2019-2023): 8,173 filas
Validación (2024): 1,987 filas
Prueba (2025): 1,998 filas


## 3.Preparar predictores (X) y objetivo (y)

Separamos lo que el modelo usa para predecir (X) de lo que queremos predecir (y,
el rendimiento).

- **Numéricos:** año, semestre y el rendimiento del pasado (rezago).
- **Categórico:** departamento, sistema productivo.

In [48]:
NUMERICAS = ["anio", "semestre_b", "rendimiento_rezago"]
CATEGORICAS = ["departamento", "sistema_productivo"]
OBJETIVO = "rendimiento"

def separar_X_y(datos):
    X = datos[NUMERICAS + CATEGORICAS]
    y = datos[OBJETIVO]
    return X, y

X_train, y_train = separar_X_y(train)
X_val,   y_val   = separar_X_y(val)
X_test,  y_test  = separar_X_y(test)

### Verificación: categorías presentes en entrenamiento

El codificador aprende los departamentos y sistemas usando solo el entrenamiento.
Confirmamos que no haya ninguna categoría en validación o prueba que falte en
entrenamiento. Si faltara, el modelo estaría prediciendo un territorio del que no
vio ningún ejemplo, y habría que decidir cómo tratarlo.

In [49]:
# Verificación: ¿todos los departamentos y sistemas de val/test están en train?
# (Si alguno faltara, el codificador fallaría y habría que decidir cómo tratarlo.)
dep_train = set(train["departamento"])
faltan_val  = set(val["departamento"])  - dep_train
faltan_test = set(test["departamento"]) - dep_train

print("Departamentos no vistos en entrenamiento:")
print(f"  en validación: {faltan_val or 'ninguno'}")
print(f"  en prueba    : {faltan_test or 'ninguno'}")

Departamentos no vistos en entrenamiento:
  en validación: ninguno
  en prueba    : ninguno


In [50]:
# Codificador deja las numéricas como están y convierte las categóricas a one-hot.
codificador = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), CATEGORICAS),
    ],
    remainder="passthrough",   # las numéricas pasan sin cambios
)

# Se AJUSTA solo con entrenamiento, y se APLICA a los tres conjuntos
X_train_cod = codificador.fit_transform(X_train)
X_val_cod   = codificador.transform(X_val)
X_test_cod  = codificador.transform(X_test)

print(f"Predictores tras codificar: {X_train_cod.shape[1]} columnas")
print(f"  (numéricas: {len(NUMERICAS)}, + one-hot de departamento y sistema)")

Predictores tras codificar: 37 columnas
  (numéricas: 3, + one-hot de departamento y sistema)


# Modelo 1: Regresión Ridge (referencia)

Empezamos con el modelo más simple e interpretable. Ridge es una regresión lineal
con una penalización que evita que los coeficientes crezcan demasiado, lo que la
hace más estable.

Lo usamos como **referencia** ya que suponemos que los modelos más complejos deberán superar su
desempeño para justificar su mayor complejidad. Si no lo logran, significa que el
problema es mayormente lineal.

In [51]:
# Pipeline: primero codifica las variables, luego aplica Ridge.
# Juntar codificación y modelo en un pipeline evita errores y asegura que
# val y test reciban exactamente el mismo tratamiento que train.
ridge = Pipeline(steps=[
    ("codificador", codificador),
    ("modelo", Ridge(alpha=1.0, random_state=SEMILLA)),
])

# Entrenamos con los datos de entrenamiento
ridge.fit(X_train, y_train)

# Predecimos sobre validación (2024)
pred_val = ridge.predict(X_val)

print("Ridge entrenado.")
print(f"Predicciones generadas para validación: {len(pred_val):,}")
print(f"\nEjemplo (primeras 5):")
for real, pred in zip(y_val[:5], pred_val[:5]):
    print(f"  real: {real:.2f} t/ha  |  predicho: {pred:.2f} t/ha")

Ridge entrenado.
Predicciones generadas para validación: 1,987

Ejemplo (primeras 5):
  real: 2.00 t/ha  |  predicho: 1.54 t/ha
  real: 2.00 t/ha  |  predicho: 1.56 t/ha
  real: 1.00 t/ha  |  predicho: 1.09 t/ha
  real: 1.00 t/ha  |  predicho: 1.11 t/ha
  real: 1.20 t/ha  |  predicho: 1.18 t/ha


## Evaluación: métricas de error

Medimos qué tan buenas son las predicciones con tres métricas estándar de regresión:

- **RMSE**: el error promedio, en las mismas unidades del rendimiento (t/ha). Penaliza más los errores grandes.
- **MAE**: el error promedio en t/ha.
- **R²**: qué porcentaje de la variación del rendimiento explica el modelo.

In [52]:
def evaluar(y_real, y_pred, nombre=""):
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    mae  = mean_absolute_error(y_real, y_pred)
    r2   = r2_score(y_real, y_pred)
    print(f"{nombre}")
    print(f"  RMSE: {rmse:.3f} t/ha")
    print(f"  MAE : {mae:.3f} t/ha")
    print(f"  R²  : {r2:.3f}")
    return {"rmse": rmse, "mae": mae, "r2": r2}

# Evaluamos Ridge sobre validación
metricas_ridge = evaluar(y_val, pred_val, "Ridge — validación 2024")

Ridge — validación 2024
  RMSE: 0.674 t/ha
  MAE : 0.447 t/ha
  R²  : 0.770


## ¿Qué variables pesan más? (coeficientes del Ridge)

Ridge asigna un coeficiente a cada variable: cuánto y en qué dirección influye en el
rendimiento. Para compararlos de forma justa, estandarizamos las variables numéricas
(las dejamos en la misma escala); si no, el rezago y el año no serían comparables por
tener magnitudes muy distintas.

Un coeficiente positivo empuja el rendimiento hacia arriba; uno negativo, hacia abajo.
El tamaño (sin signo) indica la fuerza.

## Desempeño de Ridge

- **RMSE = 0.674 t/ha:** el error típico ronda 0.67 toneladas por hectárea.
- **MAE = 0.447 t/ha:** en la mayoría de los casos, la predicción se desvía alrededor
  de 0.45 toneladas del valor real.
- **R² = 0.770:** el modelo explica el 77 % de la variación del rendimiento.

Para ser el modelo más simple, Ridge ofrece un desempeño sólido: con esencialmente una
variable fuerte (el rezago), ya explica el 77 % de la variación. Esto confirma que el
rendimiento pasado es un buen predictor del futuro y fija una referencia exigente que
los modelos más complejos deben superar.

In [53]:
# Codificador para interpretar: estandariza numéricas + one-hot en categóricas
codificador_interpretable = ColumnTransformer(transformers=[
    ("num", StandardScaler(), NUMERICAS),
    ("cat", OneHotEncoder(), CATEGORICAS),
])

ridge_interpretable = Pipeline(steps=[
    ("codificador", codificador_interpretable),
    ("modelo", Ridge(alpha=1.0, random_state=SEMILLA)),
])
ridge_interpretable.fit(X_train, y_train)

# Extraemos los coeficientes con el nombre de cada variable
nombres = ridge_interpretable.named_steps["codificador"].get_feature_names_out()
coeficientes = ridge_interpretable.named_steps["modelo"].coef_

importancia = (pd.DataFrame({"variable": nombres, "coeficiente": coeficientes})
                 .assign(peso=lambda d: d["coeficiente"].abs())
                 .sort_values("peso", ascending=False))

print("Variables con mayor peso:")
print(importancia[["variable", "coeficiente"]].head(12).to_string(index=False))

Variables con mayor peso:
                                                                  variable  coeficiente
                                                   num__rendimiento_rezago     0.903010
                                                    cat__departamento_Meta     0.764017
                                       cat__sistema_productivo_tradicional    -0.576119
                                       cat__sistema_productivo_tecnificado     0.576119
                                         cat__departamento_Valle del Cauca     0.568420
cat__departamento_Archipiélago de San Andrés, Providencia y Santa Catalina    -0.457247
                                                cat__departamento_Casanare     0.417159
                                               cat__departamento_Atlántico    -0.385611
                                                   cat__departamento_Chocó    -0.374773
                                                 cat__departamento_Quindío     0.347510
      

- **El rezago**: es la variable más fuerte (coeficiente ~0.90), indica que el rendimiento pasado es el mejor predictor del futuro en este conjunto de datos.

- **El sistema productivo**: tecnificado empuja hacia arriba (+0.58) y tradicional hacia abajo (−0.58). Coincide con el EDA, donde el tecnificado rendía más del doble.

- **Departamentos**: Algunos departamentos tienen peso propio: Meta (+0.76), Valle del Cauca (+0.57), Casanare, Tolima y Quindío hacia arriba; San Andrés, Atlántico, Chocó y Nariño hacia abajo.


## Registro en MLflow: Ridge

Envolvemos el entrenamiento en un *run* de MLflow. Todo lo que ocurra dentro del
bloque `with mlflow.start_run(...)` queda asociado a ese run: los parámetros que
usamos, las métricas que obtuvimos y el modelo entrenado. Esto nos permite comparar
este modelo con los siguientes en la interfaz de MLflow.

In [54]:
with mlflow.start_run(run_name=f"ridge_{TIPO_REZAGO}"):

    # Parámetros: la configuración que usamos
    mlflow.log_param("modelo", "Ridge")
    mlflow.log_param("tipo_rezago", TIPO_REZAGO)
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("anios_entrenamiento", f"{ANIOS_ENTRENAMIENTO[0]}-{ANIOS_ENTRENAMIENTO[-1]}")
    mlflow.log_param("anio_validacion", ANIO_VALIDACION)

    # Métricas: las que ya calculamos con evaluar()
    mlflow.log_metric("rmse_val", metricas_ridge["rmse"])
    mlflow.log_metric("mae_val", metricas_ridge["mae"])
    mlflow.log_metric("r2_val", metricas_ridge["r2"])

    # El modelo entrenado (el pipeline completo: codificador + Ridge)
    mlflow.sklearn.log_model(ridge, name="modelo")

print("Run de Ridge registrado en MLflow.")

🏃 View run ridge_estacional at: http://3.80.56.180:5000/#/experiments/1/runs/80bad5362a744ad1b27d47d2296bf7f6
🧪 View experiment at: http://3.80.56.180:5000/#/experiments/1
Run de Ridge registrado en MLflow.


# Modelo 2: Random Forest

Random Forest combina muchos árboles de decisión entrenados sobre muestras distintas
de los datos y promedia sus resultados. Esto le permite capturar **relaciones no
lineales e interacciones** entre variables que un modelo lineal como Ridge no puede.

Si logra superarlo, significa que hay estructura no lineal que vale la pena aprovechar. Si apenas lo iguala, confirma que el problema es mayormente lineal.

Usamos el mismo codificador (one-hot para las categóricas), aunque los árboles no
necesitan estandarizar las variables numéricas.

In [55]:
rf = Pipeline(steps=[
    ("codificador", codificador),
    ("modelo", RandomForestRegressor(
        n_estimators=200,       # número de árboles
        max_depth=None,         # sin límite de profundidad (se controla con los otros parámetros)
        min_samples_leaf=5,     # mínimo de observaciones por hoja: frena el sobreajuste
        random_state=SEMILLA,
        n_jobs=-1,              # usa todos los núcleos disponibles
    )),
])

rf.fit(X_train, y_train)
pred_val_rf = rf.predict(X_val)

In [56]:
metricas_rf = evaluar(y_val, pred_val_rf, f"Random Forest ({TIPO_REZAGO}) — validación 2024")

Random Forest (estacional) — validación 2024
  RMSE: 0.623 t/ha
  MAE : 0.329 t/ha
  R²  : 0.804


## Desempeño sobre RF

- **RMSE = 0.623 t/ha:** el error típico ronda 0.6 toneladas por hectárea.
- **MAE = 0.329 t/ha:** en la mayoría de los casos, la predicción se desvía alrededor
  de un tercio de tonelada del valor real. Es el más bajo de los tres modelos.
- **R² = 0.804:** el modelo explica el 80 % de la variación del rendimiento.

Random Forest mejora claramente al Ridge, sobre todo en el MAE (de 0.447 a 0.329), lo
que muestra que capturar relaciones no lineales ayuda en los casos típicos. La brecha
entre el RMSE y el MAE indica que, aun así, algunos valores extremos siguen costándole.

## Registro en MLflow: Random Forest

In [57]:
with mlflow.start_run(run_name=f"random_forest_{TIPO_REZAGO}"):

    mlflow.log_param("modelo", "RandomForest")
    mlflow.log_param("tipo_rezago", TIPO_REZAGO)
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("min_samples_leaf", 5)
    mlflow.log_param("max_depth", "None")
    mlflow.log_param("anios_entrenamiento", f"{ANIOS_ENTRENAMIENTO[0]}-{ANIOS_ENTRENAMIENTO[-1]}")
    mlflow.log_param("anio_validacion", ANIO_VALIDACION)

    mlflow.log_metric("rmse_val", metricas_rf["rmse"])
    mlflow.log_metric("mae_val", metricas_rf["mae"])
    mlflow.log_metric("r2_val", metricas_rf["r2"])

    mlflow.sklearn.log_model(rf, name="modelo", serialization_format="pickle")

print("Run de Random Forest registrado en MLflow.")

2026/11/07 11:15:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run random_forest_estacional at: http://3.80.56.180:5000/#/experiments/1/runs/187eb53a76824f4884cae79bc39c2a6e
🧪 View experiment at: http://3.80.56.180:5000/#/experiments/1
Run de Random Forest registrado en MLflow.


# Modelo 3: XGBoost

XGBoost también usa árboles, pero de forma distinta a Random Forest. En vez de
construirlos en paralelo y promediarlos, los construye **en secuencia**: cada árbol
nuevo se concentra en corregir los errores que dejaron los anteriores. Así va
afinando la predicción paso a paso.

Esta técnica (llamada *boosting*) suele lograr el mejor desempeño, pero es más
sensible a su configuración y más propensa a sobreajustar si no se controla.

In [58]:
xgb = Pipeline(steps=[
    ("codificador", codificador),
    ("modelo", XGBRegressor(
        n_estimators=300,        # número de árboles (pasos de corrección)
        learning_rate=0.05,      # cuánto corrige cada árbol: bajo = más gradual y estable
        max_depth=5,             # profundidad de cada árbol: frena el sobreajuste
        subsample=0.8,           # cada árbol usa el 80% de los datos: añade robustez
        colsample_bytree=0.8,    # cada árbol usa el 80% de las variables
        random_state=SEMILLA,
        n_jobs=-1,
    )),
])

xgb.fit(X_train, y_train)
pred_val_xgb = xgb.predict(X_val)

In [59]:
metricas_xgb = evaluar(y_val, pred_val_xgb, f"XGBoost ({TIPO_REZAGO}) — validación 2024")

XGBoost (estacional) — validación 2024
  RMSE: 0.612 t/ha
  MAE : 0.331 t/ha
  R²  : 0.810


### Desempeño de XGBoost

- **RMSE = 0.612 t/ha:** el error típico ronda 0.6 toneladas por hectárea. Es el más
  bajo de los tres modelos.
- **MAE = 0.331 t/ha:** en la mayoría de los casos, la predicción se desvía alrededor
  de un tercio de tonelada del valor real.
- **R² = 0.810:** el modelo explica el 81 % de la variación del rendimiento.

La brecha entre el RMSE (0.612) y el MAE (0.331) indica que el modelo acierta bien en
los casos típicos, pero algunos valores extremos elevan el error promedio.

## Registro en MLflow: XGBoost

In [60]:
with mlflow.start_run(run_name=f"xgboost_{TIPO_REZAGO}"):

    mlflow.log_param("modelo", "XGBoost")
    mlflow.log_param("tipo_rezago", TIPO_REZAGO)
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("subsample", 0.8)
    mlflow.log_param("colsample_bytree", 0.8)
    mlflow.log_param("anios_entrenamiento", f"{ANIOS_ENTRENAMIENTO[0]}-{ANIOS_ENTRENAMIENTO[-1]}")
    mlflow.log_param("anio_validacion", ANIO_VALIDACION)

    mlflow.log_metric("rmse_val", metricas_xgb["rmse"])
    mlflow.log_metric("mae_val", metricas_xgb["mae"])
    mlflow.log_metric("r2_val", metricas_xgb["r2"])

    mlflow.sklearn.log_model(xgb, name="modelo", serialization_format="pickle")

print("Run de XGBoost registrado en MLflow.")

2026/11/07 11:15:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run xgboost_estacional at: http://3.80.56.180:5000/#/experiments/1/runs/dd1beb4cd7f34d5a9b46214c845f0f1a
🧪 View experiment at: http://3.80.56.180:5000/#/experiments/1
Run de XGBoost registrado en MLflow.
